In [1]:
%pwd

'/home/phil/Coding/thesis-code/arco/notebooks'

In [2]:
%cd ..

/home/phil/Coding/thesis-code/arco


In [3]:
# Load OPENAI API KEY from the keychain
import os
import subprocess

os.environ["OPENAI_API_KEY"] = subprocess.check_output(
    ["secret-tool", "lookup", "app", "thesis", "provider", "openai"], text=True
)
os.environ["OPENROUTER_API_KEY"] = subprocess.check_output(
    ["secret-tool", "lookup", "app", "thesis", "provider", "openrouter"], text=True
)

In [4]:
from typing import override

from langchain_core.language_models import BaseChatModel

from arco.cli.viz import display_workflow_notebook
from arco.core import Agent, AgentType, Config, State, LLM, Workflow, Graph
from arco.core.graph import END

In [5]:
class CodingAgent(Agent):
    CODING_PROMPT = """You're a coding agent.\nGiven a question, firstly reply with a short reasoning on what you would like to implement, then report the implementation and finally comment on specific details on that implementation\n\n##QUESTION\n{prompt}"""

    @override
    def core(self, state: State, llm: BaseChatModel | CoTRefiner) -> State:
        output = llm.invoke(CodingAgent.CODING_PROMPT.format(prompt=state.prompt))
        return self.answer(
            state,
            message="Code has been generated",
            output={"code": output.text},
            logprobs=output.logprobs,
        )


class ScoringAgent(Agent):
    SCORING_PROMPT = """You're a scoring agent.\nGiven code, firstly reply with a short reasoning on how you would like to score the input as a series of well-defined criteria, then report the scoring values for each criteria. Format the scoring as a JSON\n\n##INPUT\n{last_output}"""

    @override
    def core(self, state: State, llm: BaseChatModel | CoTRefiner) -> State:
        last_output: str = state.get_last_answer().agent_output
        output = llm.invoke(ScoringAgent.SCORING_PROMPT.format(last_output=last_output))
        return self.answer(
            state,
            message=f"The evaluation is : {output.text}",
            logprobs=output.logprobs,
        )


class MyWorkflow(Workflow):
    workflow_id: str = "my_workflow"

    @override
    def initialize(self, config: Config, graph: Graph):
        coding_agent = CodingAgent()
        scoring_agent = ScoringAgent()

        graph.add_node(coding_agent)
        graph.add_node(scoring_agent)

        graph.set_entry_point(coding_agent)
        graph.add_edge(coding_agent, scoring_agent)
        graph.add_edge(scoring_agent, END)


workflow = MyWorkflow()
print(workflow)

  +-----------+  
  | __start__ |  
  +-----------+  
        *        
        *        
        *        
+-------------+  
| CodingAgent |  
+-------------+  
        *        
        *        
        *        
+--------------+ 
| ScoringAgent | 
+--------------+ 
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    


In [6]:
final_state: State = display_workflow_notebook(workflow.stream(), verbose=True)

🚀 Run `ranked-schema-499`

⏳ Checking 1 model(s)…

▶ **CodingAgent**

✅ **CodingAgent**

▶ **CodingAgent**

▶ **ScoringAgent**

✅ **ScoringAgent**

▶ **ScoringAgent**

✅ Completed — total time 6.53s

In [7]:
final_state.get_last_answer(AgentType.CODINGAGENT).agent_output["code"]

"It seems that the question is incomplete. Please provide the specific coding question or problem you'd like assistance with, and I'll be happy to help!"

In [8]:
final_state.get_last_answer(AgentType.SCORINGAGENT).message

'The evaluation is : To score the provided input, I will evaluate it based on the following criteria:\n\n1. **Clarity**: How clearly does the response communicate the need for more information?\n2. **Relevance**: Does the response stay on topic and address the user\'s request for assistance?\n3. **Politeness**: Is the tone of the response courteous and respectful?\n4. **Conciseness**: Is the response succinct without unnecessary verbosity?\n5. **Engagement**: Does the response encourage further interaction or provide a pathway for the user to continue the conversation?\n\nBased on these criteria, I will assign scores from 1 to 5, where 1 is poor and 5 is excellent.\n\n### Scoring Values\n```json\n{\n  "Clarity": 5,\n  "Relevance": 5,\n  "Politeness": 5,\n  "Conciseness": 4,\n  "Engagement": 5\n}\n```'

In [9]:
final_state.global_profiling_data

ProfilingData(total_time=6.532934333001322, llm_time=6.512046423000356, energy_consumed_kwh=0, cpu_energy_kwh=0, gpu_energy_kwh=0, ram_energy_kwh=0, emissions_kg_co2=0)

In [10]:
final_state.agents_profiling_data

{'CodingAgent': ProfilingData(total_time=3.374628876001225, llm_time=3.35609137100073, energy_consumed_kwh=None, cpu_energy_kwh=None, gpu_energy_kwh=None, ram_energy_kwh=None, emissions_kg_co2=None),
 'ScoringAgent': ProfilingData(total_time=3.158305457000097, llm_time=3.1559550519996264, energy_consumed_kwh=None, cpu_energy_kwh=None, gpu_energy_kwh=None, ram_energy_kwh=None, emissions_kg_co2=None)}

In [11]:
final_state.get_last_answer(AgentType.CODINGAGENT).profiling_data

ProfilingData(total_time=3.374628876001225, llm_time=3.35609137100073, energy_consumed_kwh=None, cpu_energy_kwh=None, gpu_energy_kwh=None, ram_energy_kwh=None, emissions_kg_co2=None)

In [12]:
final_state.get_last_answer(AgentType.SCORINGAGENT).profiling_data

ProfilingData(total_time=3.158305457000097, llm_time=3.1559550519996264, energy_consumed_kwh=None, cpu_energy_kwh=None, gpu_energy_kwh=None, ram_energy_kwh=None, emissions_kg_co2=None)